# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://mlcommons-croissant.readthedocs.io/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Citation:** Kulka, M., Rodriguez Miera, S. and Marcet-Palacios, M. 2026. Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Don't subscript, keep as one object
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
List all available record sets and their fields by their `@id`.

In [ ]:
# List record sets by their @id
# It is recommended to use metadata to discover record set and field IDs

print("Available Record Sets:")
for rs in metadata.record_sets:
    print(f"- Record Set name: {rs.name} | @id: {rs.id}")

# Show fields for each record set, referenced by their @id
for rs in metadata.record_sets:
    print(f"\nFields in Record Set {rs.name} (@id: {rs.id}):")
    for field in rs.fields:
        print(f"    - Field: {field.name} | @id: {field.id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Replace the record set and field `@id`s below with those relevant to the data you want to explore (see the above overview for valid values).

In [ ]:
# List all available record set IDs
record_set_ids = [rs.id for rs in metadata.record_sets]
print("Record Set IDs:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    # Load all records from this record set
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records from record set {record_set_id}")

# For demonstration, pick the first record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nFields (columns) for {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    print("\nSample data:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply typical EDA: filter records based on a numeric field, normalize, and group by categorical fields. All field names are referenced by their `@id` as surfaced above.

In [ ]:
# For analysis, choose a numeric and a grouping field.
# Use their @id (from the overview section).

if main_record_set_id:
    df = dataframes[main_record_set_id].copy()
    print(f"DataFrame shape: {df.shape}")
    # Infer candidate numeric fields
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if len(numeric_fields) == 0:
        # Try to force some columns numeric if available
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                pass
        numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"\nSelected numeric field for filtering: {numeric_field}")
        # Choose a threshold (mean for demo)
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Records where {numeric_field} > {threshold:.2f} : {len(filtered_df)}")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}':")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by first non-numeric field
        group_fields = [col for col in df.columns if col not in numeric_fields]
        group_field = group_fields[0] if group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped average {numeric_field} by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric field found for analysis in this record set.")

## 5. Visualization
Visualize data distributions and relationships between fields in the dataset using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_fields:
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        # Show means by group
        plt.figure(figsize=(10,4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} grouped by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Load Croissant-format metadata from a URL and explore key attributes
- Enumerate record sets and fields by their `@id`
- Extract specific record sets and perform EDA with `pandas` and `mlcroissant`
- Filter, normalize, group, and visualize domain-specific protein abundance data

> Explore additional fields and record sets for deeper biological or experimental insights. For further reproducibility, cite the dataset as described in the cell above.

This notebook was generated using the Croissant schema at https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json.

For more information about `mlcroissant`, see the [official documentation](https://mlcommons-croissant.readthedocs.io/) or [FAIR² project](https://sen.science/doi/10.71728/senscience.vq4a-28xa/).